# CSCI 4150 ? Meeting 18 ? Hybrid AI with Database Evidence
**Learn + Reverse Engineer.** This experiment uses the project?s local `medications.db` as its source of truth. It combines structured database records with lexical query matching; it needs no API key, login, package installation, or model download. This is an inspectable retrieval experiment, not a generative model or embedding search.

## 1 ? Fixed four-source test set
This controlled experiment uses exactly four evidence records: **S1** the Kaggle row for warfarin + ibuprofen; **S2** the Kaggle row for warfarin + acetaminophen, a deliberately tempting but ineligible distractor; **S3** the MedlinePlus warfarin page; and **S4** the MedlinePlus ibuprofen page. The two Kaggle rows are read by exact drug-pair match from the local SQLite database. The MedlinePlus evidence excerpts below are fixed and linked to their source pages.

Change only `QUERY` to try different prompt wording against this same four-record set. The controlled-intervention cells then change one eligibility condition at a time while keeping the query fixed.

In [7]:
import re
import sqlite3
from pathlib import Path

TOKEN_PATTERN = re.compile(r"[a-z0-9]+", re.IGNORECASE)
STOP_WORDS = {
    "a", "about", "and", "are", "can", "does", "for", "from", "how",
    "is", "of", "or", "the", "to", "what", "which", "with",
    "interaction", "interactions", "interact", "interacts", "drug", "drugs",
    "medicine", "medications", "between", "occur", "occurs", "involve",
    "involves", "involving",
}
REQUIRED_TABLES = {"drugs", "interactions"}
MEDLINEPLUS_WARFARIN_URL = "https://medlineplus.gov/druginfo/meds/a682277.html"
MEDLINEPLUS_IBUPROFEN_URL = "https://medlineplus.gov/druginfo/meds/a682159.html"

search_dirs = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
DB_PATH = next((directory / "medications.db" for directory in search_dirs
                if (directory / "medications.db").is_file()), None)
if DB_PATH is None:
    raise FileNotFoundError(
        "Could not find medications.db from the notebook's working directory. "
        "Open/run this notebook with the project folder as the working directory."
    )

connection = sqlite3.connect(DB_PATH.resolve().as_uri() + "?mode=ro", uri=True)
try:
    tables = {row[0] for row in connection.execute(
        "SELECT name FROM sqlite_master WHERE type = 'table'"
    )}
    missing_tables = REQUIRED_TABLES - tables
    if missing_tables:
        raise RuntimeError(f"Database is missing required tables: {sorted(missing_tables)}")

    def load_pair(drug_a, drug_b):
        rows = connection.execute("""
            SELECT i.interaction_id, i.severity, a.name, b.name,
                   i.mechanism, i.effect, i.safer_alternative, i.rationale, i.reference
            FROM interactions AS i
            JOIN drugs AS a ON a.drug_id = i.drug_a_id
            JOIN drugs AS b ON b.drug_id = i.drug_b_id
            WHERE (a.name = ? COLLATE NOCASE AND b.name = ? COLLATE NOCASE)
               OR (a.name = ? COLLATE NOCASE AND b.name = ? COLLATE NOCASE)
            ORDER BY i.interaction_id
        """, (drug_a, drug_b, drug_b, drug_a)).fetchall()
        if len(rows) != 1:
            raise RuntimeError(
                f"Expected exactly one database row for {drug_a} + {drug_b}; found {len(rows)}."
            )
        return rows[0]

    s1_row = load_pair("Warfarin", "Ibuprofen")
    s2_row = load_pair("Warfarin", "Acetaminophen")
finally:
    connection.close()

def format_interaction(row):
    record_id, severity, drug_a, drug_b, mechanism, effect, alternative, rationale, reference = row
    fields = [
        ("Local interaction ID", record_id),
        ("Severity", severity), ("Drug A", drug_a), ("Drug B", drug_b),
        ("Mechanism", mechanism), ("Effect", effect),
        ("Safer alternative", alternative), ("Rationale", rationale),
        ("Dataset reference", reference),
    ]
    text = "\n".join(f"{label}: {value}" for label, value in fields
                       if value is not None and str(value).strip())
    return text

DOCS = [
    {
        "id": "S1", "source": "Kaggle DDI row", "title": "Warfarin + Ibuprofen",
        "text": format_interaction(s1_row), "eligible": True,
        "eligibility_reason": "Exact drug pair asked about; direct interaction record.",
        "url": "Local project database row for warfarin + ibuprofen",
    },
    {
        "id": "S2", "source": "Kaggle DDI row", "title": "Warfarin + Acetaminophen",
        "text": format_interaction(s2_row), "eligible": False,
        "eligibility_reason": "Different pair from the requested warfarin + ibuprofen question.",
        "url": "Local project database row for warfarin + acetaminophen",
    },
    {
        "id": "S3", "source": "MedlinePlus", "title": "Warfarin drug information",
        "text": (
            "The following nonprescription or herbal products may interact with warfarin: "
            "aspirin and nonsteroidal anti-inflammatory drugs (NSAIDs) such as ibuprofen "
            "or naproxen. Tell your doctor and pharmacist before taking them; do not start "
            "them while taking warfarin without discussing with your healthcare provider."
        ), "eligible": True,
        "eligibility_reason": "Authoritative page directly names warfarin and ibuprofen.",
        "url": MEDLINEPLUS_WARFARIN_URL,
    },
    {
        "id": "S4", "source": "MedlinePlus", "title": "Ibuprofen drug information",
        "text": (
            "NSAIDs such as ibuprofen may cause ulcers, bleeding, or holes in the "
            "esophagus, stomach, or intestine. Tell your doctor if you take anticoagulants "
            "('blood thinners')."
        ), "eligible": True,
        "eligibility_reason": "Relevant safety context, but does not name warfarin specifically.",
        "url": MEDLINEPLUS_IBUPROFEN_URL,
    },
]
assert [document["id"] for document in DOCS] == ["S1", "S2", "S3", "S4"]
assert len(DOCS) == 4

QUERY = "What are the risks of taking warfarin and ibuprofen together?"

def words(text):
    return TOKEN_PATTERN.findall(text.casefold())

def query_terms(query):
    return {word for word in words(query) if len(word) > 2 and word not in STOP_WORDS}

def score(query, document):
    terms = query_terms(query)
    title_words = set(words(document["title"]))
    text_words = set(words(document["text"]))
    return sum(2 if term in title_words else 1
               for term in terms if term in title_words or term in text_words)

def rank(query, docs=DOCS, limit=4, eligible_only=True):
    if limit < 1:
        raise ValueError("limit must be greater than zero")
    ranked = []
    for document in docs:
        if eligible_only and not document["eligible"]:
            continue
        value = score(query, document)
        if value:
            ranked.append({**document, "score": value})
    return sorted(ranked, key=lambda row: (-row["score"], row["id"]))[:limit]

def show_results(query, docs=DOCS, eligible_only=True, limit=4):
    results = rank(query, docs=docs, limit=limit, eligible_only=eligible_only)
    if not results:
        print("No matching eligible evidence was retrieved; do not infer an answer.")
        return results
    for row in results:
        eligibility = "eligible" if row["eligible"] else "INELIGIBLE (intervention override)"
        print(f"[{row['id']}] {row['source']} | score={row['score']} | {eligibility}")
        print(f"  {row['title']}: {row['text']}")
        print(f"  Source: {row['url']}")
    return results

print(f"Database: {DB_PATH}")
print("Fixed evidence set: S1, S2, S3, S4 (exactly 4 records)")
for document in DOCS:
    print(f"{document['id']}: {document['title']} | eligible={document['eligible']}")
print("Change only QUERY to test query wording against this fixed set.")

Database: C:\Users\skyvu\Documents\CSCI_2600\S26\Med-Interaction-and-information\medications.db
Fixed evidence set: S1, S2, S3, S4 (exactly 4 records)
S1: Warfarin + Ibuprofen | eligible=True
S2: Warfarin + Acetaminophen | eligible=False
S3: Warfarin drug information | eligible=True
S4: Ibuprofen drug information | eligible=True
Change only QUERY to test query wording against this fixed set.


**Checkpoint A:** Predict the ranking before running retrieval. Which sources directly answer the exact pair question? Which are only contextual or distractor evidence?

## 2 ? Baseline retrieval (ineligible records excluded)

In [8]:
baseline = show_results(QUERY, eligible_only=True)
print("Retrieved IDs:", [row["id"] for row in baseline])

[S1] Kaggle DDI row | score=4 | eligible
  Warfarin + Ibuprofen: Local interaction ID: 1
Severity: Major
Drug A: Warfarin
Drug B: Ibuprofen
Mechanism: Increased bleeding risk (antiplatelet effect, GI irritation)
Effect: Increased risk of serious GI bleeding
Safer alternative: Acetaminophen (Paracetamol)
Rationale: No antiplatelet effect; safer for bleeding risk.
Dataset reference: Knox C, Wilson M, Klinger CM, et al. DrugBank 6.0: the DrugBank Knowledgebase for 2024. Nucleic Acids Res. 2024 Jan 5;52(D1):D1265-D1275. doi: 10.1093/nar/gkad976., Micromedex
  Source: Local project database row for warfarin + ibuprofen
[S3] MedlinePlus | score=4 | eligible
  Warfarin drug information: The following nonprescription or herbal products may interact with warfarin: aspirin and nonsteroidal anti-inflammatory drugs (NSAIDs) such as ibuprofen or naproxen. Tell your doctor and pharmacist before taking them; do not start them while taking warfarin without discussing with your healthcare provider.
  S

**Checkpoint B:** Did S1 and S3 appear? Is S4 direct evidence for this exact drug pair, or general safety context?

## 3 ? Controlled intervention A ? let the tempting distractor in

In [9]:
print("Baseline: exclude ineligible S2")
baseline = show_results(QUERY, eligible_only=True)
print("\nIntervention: override eligibility and include S2; query and records stay fixed")
with_s2 = show_results(QUERY, eligible_only=False)
print("\nBaseline IDs:", [row["id"] for row in baseline])
print("Intervention IDs:", [row["id"] for row in with_s2])

Baseline: exclude ineligible S2
[S1] Kaggle DDI row | score=4 | eligible
  Warfarin + Ibuprofen: Local interaction ID: 1
Severity: Major
Drug A: Warfarin
Drug B: Ibuprofen
Mechanism: Increased bleeding risk (antiplatelet effect, GI irritation)
Effect: Increased risk of serious GI bleeding
Safer alternative: Acetaminophen (Paracetamol)
Rationale: No antiplatelet effect; safer for bleeding risk.
Dataset reference: Knox C, Wilson M, Klinger CM, et al. DrugBank 6.0: the DrugBank Knowledgebase for 2024. Nucleic Acids Res. 2024 Jan 5;52(D1):D1265-D1275. doi: 10.1093/nar/gkad976., Micromedex
  Source: Local project database row for warfarin + ibuprofen
[S3] MedlinePlus | score=4 | eligible
  Warfarin drug information: The following nonprescription or herbal products may interact with warfarin: aspirin and nonsteroidal anti-inflammatory drugs (NSAIDs) such as ibuprofen or naproxen. Tell your doctor and pharmacist before taking them; do not start them while taking warfarin without discussing wi

**Checkpoint C:** What changed when only the eligibility filter changed? Did S2 outrank direct evidence, appear as irrelevant context, or have no effect? Label any false inclusion. Restore the original eligibility rule after observing the result.

## 4 ? Controlled intervention B ? remove the direct-match source

In [10]:
without_s1 = [document for document in DOCS if document["id"] != "S1"]
print("Baseline: all four sources, with S2 excluded by eligibility")
show_results(QUERY, eligible_only=True)
print("\nIntervention: remove only S1; keep query and remaining source eligibility fixed")
without_s1_results = show_results(QUERY, docs=without_s1, eligible_only=True)
print("\nRemaining IDs:", [row["id"] for row in without_s1_results])

Baseline: all four sources, with S2 excluded by eligibility
[S1] Kaggle DDI row | score=4 | eligible
  Warfarin + Ibuprofen: Local interaction ID: 1
Severity: Major
Drug A: Warfarin
Drug B: Ibuprofen
Mechanism: Increased bleeding risk (antiplatelet effect, GI irritation)
Effect: Increased risk of serious GI bleeding
Safer alternative: Acetaminophen (Paracetamol)
Rationale: No antiplatelet effect; safer for bleeding risk.
Dataset reference: Knox C, Wilson M, Klinger CM, et al. DrugBank 6.0: the DrugBank Knowledgebase for 2024. Nucleic Acids Res. 2024 Jan 5;52(D1):D1265-D1275. doi: 10.1093/nar/gkad976., Micromedex
  Source: Local project database row for warfarin + ibuprofen
[S3] MedlinePlus | score=4 | eligible
  Warfarin drug information: The following nonprescription or herbal products may interact with warfarin: aspirin and nonsteroidal anti-inflammatory drugs (NSAIDs) such as ibuprofen or naproxen. Tell your doctor and pharmacist before taking them; do not start them while taking wa

**Checkpoint D:** After removing S1, can MedlinePlus still support the key point? What information is lost without the structured interaction row? This isolates the contribution of the direct-match record.

## 5 ? Query-only variation and limitations

Change only `QUERY` and rerun the retrieval cells to test wording against the same four sources. Then use the intervention cells to change one condition at a time. Record query, intervention, retrieved IDs/scores, whether evidence directly supports the response, and failure label. This is lexical retrieval over a tiny fixed evidence set, not an LLM or medical advice. S3 explicitly names both drugs; S4 gives general NSAID/anticoagulant context but does not name warfarin. Consult a qualified health professional for medication decisions.

## 6 ? Evidence log

| Trial | One change | Retrieved IDs | Evidence interpretation | Failure label / design implication |
|---|---|---|---|---|
| Baseline | None; S2 excluded | | | |
| Intervention A | Include ineligible S2 | | | |
| Intervention B | Remove S1 | | | |
| Query variant | Change only `QUERY` | | | |

A justified **REJECT / DEFER** is valid evidence. R1 = retrieval miss, R2 = evidence split or missing context, R3 = irrelevant evidence included, G1 = unsupported generation, C1 = citation mismatch, E1 = insufficient evidence.